# 🚬 Tobacco Smoking Risk Classification
**MLOps Project — Google Colab Phase**

Dataset: CDC BRFSS Tobacco Use Survey (`rows.csv`)
Target: `Risk_Level` — **High Risk (1)** / **Low Risk (0)** berdasarkan prevalensi merokok per wilayah & demografi

---
Pipeline Colab:
1. Setup & Install
2. Load Dataset
3. Exploratory Data Analysis (EDA)
4. Preprocessing
5. Baseline Modelling
6. Hyperparameter Tuning
7. Export & Download

## 1. Setup & Install Dependencies

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn joblib -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

## 2. Load Dataset

> **Pilih salah satu cara upload dataset:**

In [ ]:
# Option A: Upload langsung dari komputer
from google.colab import files
uploaded = files.upload()  # upload rows.csv
DATA_PATH = 'rows.csv'

# Option B: Mount Google Drive (uncomment jika pakai Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/MLOps/rows.csv'

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape     : {df.shape}')
print(f'Columns   : {list(df.columns)}')
print(f'\nPreview:')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
print('=== Data_Value Statistics ===')
print(df['Data_Value'].describe())
median_val = df['Data_Value'].median()
print(f'\nMedian (Threshold untuk Risk_Level): {median_val:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['Data_Value'].dropna().hist(bins=40, ax=axes[0], color='#E8553E', edgecolor='white')
axes[0].axvline(median_val, color='navy', linestyle='--', linewidth=2,
                label=f'Median = {median_val:.1f}%')
axes[0].set_title('Distribusi Prevalensi Merokok (Data_Value)', fontsize=13)
axes[0].set_xlabel('Data_Value (%)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

df.groupby('YEAR')['Data_Value'].mean().plot(
    ax=axes[1], marker='o', color='#10B981', linewidth=2)
axes[1].set_title('Rata-rata Prevalensi Merokok per Tahun', fontsize=13)
axes[1].set_ylabel('Avg Data_Value (%)')
axes[1].set_xlabel('Year')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('eda_distribusi_tren.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, color, title in zip(
    axes,
    ['Gender', 'Age', 'Education'],
    ['#3B82F6', '#8B5CF6', '#F59E0B'],
    ['per Gender', 'per Age Group', 'per Education']
):
    df.groupby(col)['Data_Value'].mean().sort_values().plot(
        kind='barh', ax=ax, color=color)
    ax.set_title(f'Rata-rata Prevalensi\n{title}', fontsize=12)
    ax.set_xlabel('Avg Data_Value (%)')

plt.tight_layout()
plt.savefig('eda_demografi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
state_avg = df.groupby('LocationAbbr')['Data_Value'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

state_avg.nlargest(10).sort_values().plot(
    kind='barh', ax=axes[0], color='#EF4444', edgecolor='white')
axes[0].set_title('Top 10 State — Prevalensi Tertinggi 🔴', fontsize=12)
axes[0].set_xlabel('Avg Data_Value (%)')

state_avg.nsmallest(10).sort_values(ascending=False).plot(
    kind='barh', ax=axes[1], color='#22C55E', edgecolor='white')
axes[1].set_title('Top 10 State — Prevalensi Terendah 🟢', fontsize=12)
axes[1].set_xlabel('Avg Data_Value (%)')

plt.tight_layout()
plt.savefig('eda_state.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['DataSource'].value_counts().plot(kind='bar', ax=axes[0], color='#6366F1', edgecolor='white')
axes[0].set_title('Distribusi DataSource')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

df['Response'].value_counts().plot(kind='bar', ax=axes[1], color='#EC4899', edgecolor='white')
axes[1].set_title('Distribusi Response (Smoking Status)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pandas as pd # Ensure pandas is imported for pd.to_numeric

def preprocess(df):
    df = df.copy()

    # Step 1: Drop kolom tidak relevan
    cols_drop = [
        'index', 'LocationDesc', 'TopicType', 'TopicDesc',
        'Data_Value_Unit', 'Data_Value_Type',
        'Data_Value_Footnote_Symbol', 'Data_Value_Footnote',
        'Data_Value_Std_Err', 'Low_Confidence_Limit', 'High_Confidence_Limit',
        'GeoLocation', 'TopicTypeId', 'TopicId', 'MeasureId',
        'StratificationID1', 'StratificationID2', 'StratificationID3',
        'StratificationID4', 'SubMeasureID', 'DisplayOrder'
    ]
    df.drop(columns=cols_drop, errors='ignore', inplace=True)
    print(f'[INFO] Setelah drop kolom       : {df.shape}')

    # --- NEW STEP: Clean and convert 'YEAR' column to numeric ---
    if 'YEAR' in df.columns and df['YEAR'].dtype == 'object':
        # Define a function to clean 'YEAR' values
        def clean_year_column(year_val):
            if isinstance(year_val, str):
                if '-' in year_val:
                    # Extract the first year from a range like '2013-2014'
                    try:
                        return int(year_val.split('-')[0])
                    except ValueError:
                        return None # Return None for unparseable strings
                else:
                    # Convert single year string to int
                    try:
                        return int(year_val)
                    except ValueError:
                        return None # Return None for unparseable strings
            return year_val # Return as is if already numeric or NaN

        df['YEAR'] = df['YEAR'].apply(clean_year_column)
        # Convert to numeric, coercing any remaining errors to NaN
        df['YEAR'] = pd.to_numeric(df['YEAR'], errors='coerce')
        # Fill NaN values in 'YEAR' (e.g., with median) if any exist after cleaning
        if df['YEAR'].isnull().any():
            median_year = df['YEAR'].median()
            df['YEAR'].fillna(median_year, inplace=True)
        df['YEAR'] = df['YEAR'].astype(int) # Convert to integer type
    print(f'[INFO] Setelah cleaning YEAR    : {df.shape}')
    # --- END NEW STEP ---

    # Step 2: Handle missing values
    df.dropna(subset=['Data_Value'], inplace=True)
    cat_cols = ['LocationAbbr', 'MeasureDesc', 'DataSource', 'Response',
                'Gender', 'Race', 'Age', 'Education']
    for col in cat_cols:
        if col in df.columns:
            df[col].fillna(df[col].mode()[0], inplace=True)
    if 'Sample_Size' in df.columns:
        df['Sample_Size'].fillna(df['Sample_Size'].median(), inplace=True)
    print(f'[INFO] Setelah handle missing   : {df.shape}')

    # Step 3: Buat target Risk_Level dari Data_Value
    median_val = df['Data_Value'].median()
    print(f'[INFO] Threshold (median)       : {median_val:.2f}%')
    df['Risk_Level'] = (df['Data_Value'] > median_val).astype(int)
    df.drop(columns=['Data_Value'], inplace=True)

    # Step 4: Encode kolom kategorik
    le = LabelEncoder()
    for col in cat_cols:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))

    # Step 5: Scale kolom numerik
    # 'YEAR' should now be numeric due to the new cleaning step
    num_cols = [c for c in ['YEAR', 'Sample_Size'] if c in df.columns]
    scaler = StandardScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols])

    print(f'[INFO] Preprocessing selesai    : {df.shape}')
    return df

df_processed = preprocess(df)
print('\n=== Risk_Level Distribution ===')
print(df_processed['Risk_Level'].value_counts())
print(df_processed['Risk_Level'].value_counts(normalize=True).map('{:.1%}'.format))
df_processed.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colors = ['#22C55E', '#EF4444']
labels = ['Low Risk (0)', 'High Risk (1)']
counts = df_processed['Risk_Level'].value_counts().sort_index()

axes[0].bar(labels, counts.values, color=colors, edgecolor='white')
axes[0].set_title('Distribusi Risk_Level')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporsi Risk_Level')

plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('tobacco_preprocessed', exist_ok=True)
df_processed.to_csv('tobacco_preprocessed/preprocessed.csv', index=False)
print('✓ Saved: tobacco_preprocessed/preprocessed.csv')
print(f'  Shape   : {df_processed.shape}')
print(f'  Columns : {list(df_processed.columns)}')

## 5. Baseline Modelling

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
)

X = df_processed.drop(columns=['Risk_Level'])
y = df_processed['Risk_Level']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {X_train.shape}')
print(f'Test size  : {X_test.shape}')

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('=== Classification Report (Baseline) ===')
print(classification_report(y_test, y_pred, target_names=['Low Risk', 'High Risk']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Low Risk', 'High Risk']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Baseline', fontsize=13)

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], color='#E8553E')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[1].set_title('ROC Curve — Baseline', fontsize=13)

plt.tight_layout()
plt.savefig('baseline_eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='#8B5CF6', edgecolor='white')
ax.set_title('Feature Importances — Baseline Model', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':      [50, 100, 200],
    'max_depth':         [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf':  [1, 2]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print('Running GridSearchCV... (estimasi 2-5 menit)')
grid_search.fit(X_train, y_train)

print(f'\n✓ Best Params  : {grid_search.best_params_}')
print(f'✓ Best F1 (CV) : {grid_search.best_score_:.4f}')

In [ ]:
best_model = grid_search.best_estimator_
y_pred_tuned  = best_model.predict(X_test)
y_proba_tuned = best_model.predict_proba(X_test)[:, 1]

print('=== Classification Report (Tuned) ===')
print(classification_report(y_test, y_pred_tuned, target_names=['Low Risk', 'High Risk']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba_tuned):.4f}')

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

metrics = {
    'Accuracy':  [accuracy_score(y_test, y_pred),  accuracy_score(y_test, y_pred_tuned)],
    'F1 Score':  [f1_score(y_test, y_pred),         f1_score(y_test, y_pred_tuned)],
    'Precision': [precision_score(y_test, y_pred),  precision_score(y_test, y_pred_tuned)],
    'Recall':    [recall_score(y_test, y_pred),     recall_score(y_test, y_pred_tuned)],
    'AUC-ROC':   [roc_auc_score(y_test, y_proba),  roc_auc_score(y_test, y_proba_tuned)],
}

df_compare = pd.DataFrame(metrics, index=['Baseline', 'Tuned']).T
print(df_compare.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
df_compare.plot(kind='bar', ax=ax, color=['#6366F1', '#F59E0B'], edgecolor='white', width=0.6)
ax.set_ylim(0, 1.1)
ax.set_title('Perbandingan Baseline vs Tuned Model', fontsize=13)
ax.set_ylabel('Score')
ax.legend(['Baseline', 'Tuned'])
ax.tick_params(axis='x', rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig('comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export & Download

In [ ]:
import joblib

joblib.dump(model,      'rf_model.pkl')
joblib.dump(best_model, 'rf_model_tuned.pkl')
df_compare.to_csv('model_metrics.csv')

print('✓ rf_model.pkl')
print('✓ rf_model_tuned.pkl')
print('✓ model_metrics.csv')
print('✓ tobacco_preprocessed/preprocessed.csv')

In [ ]:
from google.colab import files

for fname in [
    'tobacco_preprocessed/preprocessed.csv',
    'rf_model.pkl',
    'rf_model_tuned.pkl',
    'model_metrics.csv',
    'eda_distribusi_tren.png',
    'eda_demografi.png',
    'eda_state.png',
    'baseline_eval.png',
    'feature_importance.png',
    'comparison.png',
]:
    if os.path.exists(fname):
        files.download(fname)
        print(f'✓ Downloaded: {fname}')
    else:
        print(f'✗ Not found : {fname}')